# KEGG: original DREAMwalk, local run

Use a **public KEGG REST snapshot** and the authors' original DREAMwalk to compare with **Figure 2d**. MSI and Hetionet retain separate notebooks and outputs.

**Select any local Python 3 kernel and run from top to bottom.** Computation uses the separately installed Python 3.8 environment. Training and evaluation start when you run their cells.

Retain the **original teleport selector, C++ `pp=1`, EM and network traversal**. MSI selector, NS-pool and positive-output-filter variants are not installed.

**Input scope:** the paper-linked GitHub/Zenodo release does not supply the complete processed KEGG benchmark. This notebook saves and reconstructs public API responses without claiming the authors' historical snapshot. Later stages reuse the same saved inputs.


## 1. Paths, fixed parameters and paper reference

Open from the repository root, model directory or `notebooks/`. The notebook host does not need the model dependencies; scientific computation uses the displayed Python 3.8 interpreter.


In [1]:
from pathlib import Path
import sys
from IPython.display import JSON, display

sys.dont_write_bytecode = True
candidates = []
for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    candidates.extend((base, base / "00.Reproduction" / "00.DREAMwalk_reproduction"))
ROOT = next((base for base in candidates
             if (base / "src" / "run_kegg_original.py").is_file()
             and (base / "vendor" / "DREAMwalk_upstream").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("DREAMwalk project not found; open the notebook from the repository or model directory.")

sys.path.insert(0, str(ROOT / "src"))
import dreamwalk_notebook_workflow as workflow
import kegg_inputs

PYTHON = ROOT / "environment" / ".conda-env" / "bin" / "python"
RUNNER = ROOT / "src" / "run_kegg_original.py"
RUN_NAME = "kegg_seed0_original"
DATA = ROOT / "data" / "embeddings" / RUN_NAME
RESULTS = ROOT / "results" / RUN_NAME
LOGS = ROOT / "logs" / RUN_NAME
INPUTS = kegg_inputs.paths(ROOT)
EMBEDDING = DATA / "kegg_seed0_original.pkl"

SETTINGS = dict(tp_factor=0.3, cutoff=0.4, seed=0, num_walks=100,
                walk_length=10, dimension=128, window_size=4, workers=4,
                em_max_iter=5, p=1, q=1, pp=1, epochs=1, negative=5,
                sample=0.001, folds=10, cv_seed=0)
actual = workflow.runner_constants(RUNNER, "SETTINGS")["SETTINGS"]
workflow.check_parameters(actual, SETTINGS)
if actual != SETTINGS:
    raise ValueError("Notebook and runner parameter dictionaries differ.")
if not PYTHON.is_file():
    raise FileNotFoundError("Python 3.8 interpreter not found: " + str(PYTHON))

print("Project: ", ROOT)
print("Computation interpreter: ", PYTHON)
print("Run name：", RUN_NAME)
print("Embedding：", EMBEDDING)
print("Logs：", LOGS)
workflow.show_rows([dict(parameter=key, value=value) for key, value in SETTINGS.items()])

Project: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction
Computation interpreter: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/environment/.conda-env/bin/python
Run name： kegg_seed0_original
Embedding： /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/kegg_seed0_original/kegg_seed0_original.pkl
Logs： /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/logs/kegg_seed0_original


parameter,value
tp_factor,0.300000
cutoff,0.400000
seed,0
num_walks,100
walk_length,10
dimension,128
window_size,4
workers,4
em_max_iter,5
p,1


### Fixed pipeline

- Similarity cutoff `0.4`; teleport factor `τ=0.3`.
- `100` walks per active node, length `10`, `p=q=1`, at most `5` edge-type EM iterations, `4` workers.
- Original heterogeneous Skip-gram: `pp=1`, 128 dimensions, window `4`, one epoch, five negatives, subsampling `0.001`.
- Features: `drug embedding − disease embedding`; released XGBoost settings, 500 trees and depth 6.
- Evaluation: **one local seed-0 negative set × ten folds**, compared with ten paper repeats. No automatic expansion to 100 fits.
- `τ=0.3` follows the accepted demo/local setting; it is not claimed as an explicitly reported Figure 2d value.
- Keep upstream `set_seed(0)` without an extra startup hash seed. Worker RNG and four-thread C++ training are not asserted to be deterministic.

References come from **Source Data / Figure2d / B2:D11**, not plot estimates. Provenance records workbook hash, cells, ten raw rows and seeds.


In [2]:
REFERENCE = ROOT / "references" / "kegg_fig2d_dreamwalk.csv"
REFERENCE_PROVENANCE = ROOT / "references" / "kegg_fig2d_dreamwalk_provenance.json"
workflow.show_csv(REFERENCE)
display(JSON(workflow.read_json(REFERENCE_PROVENANCE), expanded=False))

kegg_fig2d_dreamwalk.csv


metric,mean
accuracy,0.888669
auroc,0.952773
aupr,0.954531


<IPython.core.display.JSON object>

## 2. Check the environment and original source

Check Python 3.8, package pins, commit `2f93723c3fbfa899d0f23c02e3459de8bfac0581` and unchanged tracked upstream files. Restore upstream Git metadata using the repository README first.


In [3]:
workflow.run_phase(ROOT, PYTHON, "run_kegg_original.py", "environment", RUN_NAME)

Running: run_kegg_original.py environment | kegg_seed0_original
{
  "packages": {
    "numpy": "1.23.3",
    "scipy": "1.9.1",
    "scikit-learn": "1.1.3",
    "networkx": "2.8.8",
    "pandas": "1.4.4",
    "tqdm": "4.64.1",
    "parmap": "1.6.0",
    "xgboost": "1.7.4"
  },
  "upstream_commit": "2f93723c3fbfa899d0f23c02e3459de8bfac0581",
  "settings": {
    "tp_factor": 0.3,
    "cutoff": 0.4,
    "seed": 0,
    "num_walks": 100,
    "walk_length": 10,
    "dimension": 128,
    "window_size": 4,
    "workers": 4,
    "em_max_iter": 5,
    "p": 1,
    "q": 1,
    "pp": 1,
    "epochs": 1,
    "negative": 5,
    "sample": 0.001,
    "folds": 10,
    "cv_seed": 0
  },
  "xgboost": {
    "base_score": 0.5,
    "booster": "gbtree",
    "eval_metric": "error",
    "objective": "binary:logistic",
    "gamma": 0,
    "learning_rate": 0.1,
    "max_depth": 6,
    "n_estimators": 500,
    "tree_method": "auto",
    "min_child_weight": 4,
    "subsample": 0.8,
    "colsample_bytree": 0.9,
    "

## 3. Download and freeze the KEGG snapshot

Download relation tables, entry lists, pathway/disease records and ATC/ICD-11 BRITE hierarchies. Save request URLs, responses, hashes and release information. Resume saved requests without refreshing completed responses.

- IDs: drugs `Dxxxxx`, diseases `Hxxxxx`, human genes `hsa:...`.
- Normalize human `hsaXXXXX` and reference `mapXXXXX` pathways by the shared five-digit identifier; audit this normalization.
- Use human genes only; exclude KO and other organisms from the human gene pool.
- Batch `link` responses do not contain every flat-file relation. Disease, drug and related-pathway information also requires entry records.
- Plain `get` requests contain at most ten entries per batch and follow the official API rate limit. Downloads may take several minutes and resume from saved responses.

This freezes **the public snapshot obtained for this run**, rather than recovering the authors' 2023 data. Save paper and actual counts side by side during preparation.


In [4]:
workflow.run_phase(ROOT, PYTHON, "run_kegg_original.py", "download", RUN_NAME)

Running: run_kegg_original.py download | kegg_seed0_original
KEGG raw cards: 408 batches cached, 0 to download (10 records/batch, at most 2.5 requests/s)


In [5]:
print("Raw response directory: ", INPUTS["raw_dir"])
if INPUTS["raw_dir"].is_dir():
    files = sorted(path for path in INPUTS["raw_dir"].rglob("*") if path.is_file())
    print("Saved file count: ", len(files))
    workflow.show_rows([dict(file=str(path.relative_to(INPUTS["raw_dir"])), bytes=path.stat().st_size)
                        for path in files[:12]])
else:
    print("Inputs are not downloaded; run the download cell above.")

Raw response directory: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/kegg_raw
Saved file count: 421


file,bytes
br08303_atc.txt,688704
br08403_icd11.txt,353551
cards/disease_H00001_2e5c8112a2ba.txt,53903
cards/disease_H00011_5ffbdc1ddbef.txt,75936
cards/disease_H00021_55183eb9babb.txt,55982
cards/disease_H00031_4b7bfd820593.txt,58023
cards/disease_H00041_b0397454d197.txt,47024
cards/disease_H00051_ea0f58035199.txt,62304
cards/disease_H00061_4088c5b6f34d.txt,64426
cards/disease_H00071_4571f691395a.txt,47459


## 4. Prepare graph, hierarchy, labels, negatives and folds

### Historical counts from Supplementary Table 4

| Node type | Paper count |
|---|---:|
| Drug | 6,008 |
| Disease | 1,963 |
| Gene | 14,496 |
| Pathway | 461 |

| Relation | Paper count | Use |
|---|---:|---|
| Drug–disease | 2,272 | Positive labels; excluded from the embedding graph |
| Disease–gene | 6,319 | Molecular graph |
| Drug–gene | 11,860 | Molecular graph |
| Gene–pathway | 43,226 | Molecular graph |
| Pathway–pathway | 2,129 | Molecular graph |
| Disease–pathway | 2,573 | Molecular graph |
| Drug–pathway | 10,274 | Molecular graph |

The reconstruction uses the **six molecular relation types in Supplementary Table 4**. The main text names only gene–pathway, drug–gene and disease–gene relations; the complete preprocessing script is unavailable. Record this explicit recipe without claiming exact historical inputs.

**Semantic inputs:** public KEGG BRITE provides drug ATC and disease ICD-11 paths connected to native KEGG IDs. The paper uses DrugBank ATC assignments, so the KEGG BRITE substitution is a recorded source difference. Unmapped entities receive no semantic paths. Save coverage, full hierarchy paths and preprocessing audits.

**Labels and evaluation:** public drug–disease associations define positives and never become embedding edges. Evaluate only pairs whose endpoints occur in the active molecular graph; audit exclusions. Sample an equal number of negatives with seed `0` from active drug–disease pairs excluding all known positives. Save negative IDs, pair order and ten-fold IDs once.

**Actual counts are defined by the saved manifest.** Do not edit edges, add self-loops or select labels to force agreement with paper counts.


In [6]:
workflow.run_phase(ROOT, PYTHON, "run_kegg_original.py", "prepare", RUN_NAME)

Running: run_kegg_original.py prepare | kegg_seed0_original
{
  "raw_positives": 5649,
  "evaluable_positives": 2720,
  "excluded_no_molecular_embedding": 2929,
  "negatives": 2720,
  "molecular_nodes": 20613
}


In [ ]:
if INPUTS["manifest"].is_file():
    manifest = workflow.read_json(INPUTS["manifest"])
    display(JSON(manifest, expanded=False))
    print("Molecular edge types：")
    workflow.show_rows(manifest.get("molecular", {}).get("edge_types", []))
    print("Semantic coverage：")
    display(JSON(manifest.get("semantic", {}), expanded=True))
    print("Table 4 and actual input counts: ")
    workflow.show_csv(INPUTS["audit_dir"] / "paper_table4_counts.csv")
    print("Input audits：", INPUTS["audit_dir"])
    excluded = INPUTS["audit_dir"] / "excluded_positive_pairs.csv"
    if excluded.is_file():
        print("Excluded positive pairs：")
        excluded_rows = workflow.read_csv(excluded)
        print("Excluded count: ", len(excluded_rows), "; displaying at most 20 rows. Full CSV: ", excluded)
        workflow.show_rows(excluded_rows[:20])
else:
    print("Inputs are not prepared; run the prepare cell first.")

config_path = RESULTS / "run_config.json"
if config_path.is_file():
    config = workflow.read_json(config_path)
    display(JSON({key: config[key] for key in ("counts", "negative_policy", "notes") if key in config},
                 expanded=True))
    print("Negative pairs：", RESULTS / "negative_pairs.csv")
    print("Fold IDs：", RESULTS / "fold_assignments.csv")
else:
    print("No saved run configuration.")

## 5. Generate semantic similarity

Use upstream IC/similarity code to generate drug–drug and disease–disease edges from ATC/ICD-11 hierarchies at cutoff `0.4`. This does not train embeddings.


In [8]:
workflow.run_phase(ROOT, PYTHON, "run_kegg_original.py", "similarity", RUN_NAME)

Running: run_kegg_original.py similarity | kegg_seed0_original
Similarity graph saved: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/kegg_seed0_original/similarity_pending.tsv
Original semantic graph saved: {'drug': 134241, 'disease': 51832}


In [9]:
marker = RESULTS / "similarity_complete.json"
if marker.is_file():
    display(JSON(workflow.read_json(marker), expanded=True))
else:
    print("Similarity graph is not complete.")

<IPython.core.display.JSON object>

## 6. Original EM → full walks → C++ `pp=1` embeddings

Call the unchanged upstream pipeline with frozen inputs/settings. Generate **100 × active molecular nodes** walks, each of length `10`.

This is usually the longest stage. Stream and save its output to `embedding.log`.

**Atomically save the completed embedding as `.pkl` and write its completion marker before XGBoost evaluation.** Completed embeddings survive an interrupted evaluation. Upstream combines EM, walks and C++ training in one call; incomplete embeddings are not accepted as checkpoints.


In [10]:
workflow.run_phase(ROOT, PYTHON, "run_kegg_original.py", "embedding", RUN_NAME)

Running: run_kegg_original.py embedding | kegg_seed0_original
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/kegg_seed0_original/similarity_graph.tsv
random seed with 0
Reading network files...
Training edge type transition matrix...
Generating paths...
Generating node embeddings...
DREAMwalk/HeterogeneousSG.cpp:328:16: warning: variable 'i' set but not used [-Wunused-but-set-variable]
  328 |   long long a, i = 0;
      |                ^
1 warning generated.
Node embeddings saved: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/logs/kegg_seed0_original/original_embedding_kqssgjzg/embedding_pending.pkl
EMBEDDING SAVED before evaluation: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/kegg_seed0_original/kegg_seed0_original.pkl


In [11]:
marker = RESULTS / "embedding_complete.json"
if marker.is_file():
    display(JSON(workflow.read_json(marker), expanded=True))
    print("Saved embedding: ", EMBEDDING)
else:
    print("Embedding is not complete. Destination: ", EMBEDDING)

<IPython.core.display.JSON object>

Saved embedding: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/kegg_seed0_original/kegg_seed0_original.pkl


## 7. XGBoost on saved pairs and folds

Read the saved embedding, pair order, negative IDs and ten-fold test IDs. Keep XGBoost parameters fixed. Save and reuse completed folds.

Accuracy uses `prob.round()`, AUROC uses predicted probabilities, and AUPR uses `average_precision_score`. Run ten folds for this seed-0 set only.


In [12]:
workflow.run_phase(ROOT, PYTHON, "run_kegg_original.py", "evaluate", RUN_NAME)

Running: run_kegg_original.py evaluate | kegg_seed0_original
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/kegg_seed0_original/kegg_seed0_original.pkl
Fold 1 {'accuracy': 0.919118, 'auroc': 0.970994, 'aupr': 0.971369}
Fold 2 {'accuracy': 0.930147, 'auroc': 0.973765, 'aupr': 0.972412}
Fold 3 {'accuracy': 0.950368, 'auroc': 0.987659, 'aupr': 0.987466}
Fold 4 {'accuracy': 0.939338, 'auroc': 0.98228, 'aupr': 0.98349}
Fold 5 {'accuracy': 0.920956, 'auroc': 0.978617, 'aupr': 0.977486}
Fold 6 {'accuracy': 0.933824, 'auroc': 0.976846, 'aupr': 0.975398}
Fold 7 {'accuracy': 0.935662, 'auroc': 0.978036, 'aupr': 0.980161}
Fold 8 {'accuracy': 0.930147, 'auroc': 0.978374, 'aupr': 0.981829}
Fold 9 {'accuracy': 0.950368, 'auroc': 0.984267, 'aupr': 0.980039}
Fold 10 {'accuracy': 0.924632, 'auroc': 0.977765, 'aupr': 0.978107}
Ten-fold evaluation saved: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reprod

In [13]:
workflow.show_csv(RESULTS / "fold_results.csv")
workflow.show_csv(RESULTS / "summary.csv")

fold_results.csv


fold,accuracy,auroc,aupr,n_train,n_test,seconds
1,0.919118,0.970994,0.971369,4896,544,6.039238
2,0.930147,0.973765,0.972412,4896,544,6.054217
3,0.950368,0.987659,0.987466,4896,544,6.097430
4,0.939338,0.982280,0.983490,4896,544,6.069550
5,0.920956,0.978617,0.977486,4896,544,6.056550
6,0.933824,0.976846,0.975398,4896,544,6.074668
7,0.935662,0.978036,0.980161,4896,544,6.083498
8,0.930147,0.978374,0.981829,4896,544,6.089439
9,0.950368,0.984267,0.980039,4896,544,6.110086
10,0.924632,0.977765,0.978107,4896,544,6.023723


summary.csv


metric,mean,SD,n
accuracy,0.933456,0.010920,10
auroc,0.978860,0.004858,10
aupr,0.978776,0.004938,10


## 8. Compare with Figure 2d

Report **local − paper** for Accuracy, AUROC and AUPR. Interpret these together with the MSI and Hetionet results and their input differences.

Paper references average ten Source Data rows; local metrics average ten folds for one negative set. Report snapshot, graph recipe, semantic coverage and excluded-label audits with performance.

Stop here. No additional seeds, parameter search or MSI patches are started automatically.


In [14]:
workflow.run_phase(ROOT, PYTHON, "run_kegg_original.py", "compare", RUN_NAME)

Running: run_kegg_original.py compare | kegg_seed0_original
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/kegg_seed0_original/summary.csv
[
  {
    "metric": "accuracy",
    "local_seed0_10fold": 0.9334558823529412,
    "paper_10repeat_mean": 0.8886688668866884,
    "local_minus_paper": 0.044787015466252744
  },
  {
    "metric": "auroc",
    "local_seed0_10fold": 0.9788602941176471,
    "paper_10repeat_mean": 0.9527729099094737,
    "local_minus_paper": 0.02608738420817336
  },
  {
    "metric": "aupr",
    "local_seed0_10fold": 0.9787756768143218,
    "paper_10repeat_mean": 0.954530956059595,
    "local_minus_paper": 0.024244720754726723
  }
]


In [15]:
workflow.show_csv(RESULTS / "paper_comparison.csv")
marker = RESULTS / "comparison_complete.json"
if marker.is_file():
    display(JSON(workflow.read_json(marker), expanded=False))
else:
    print("No final comparison; run the preceding cell after evaluation completes.")
print("Results directory: ", RESULTS)
print("Logs directory: ", LOGS)

paper_comparison.csv


metric,local_seed0_10fold,paper_10repeat_mean,local_minus_paper
accuracy,0.933456,0.888669,0.044787
auroc,0.978860,0.952773,0.026087
aupr,0.978776,0.954531,0.024245


<IPython.core.display.JSON object>

Results directory: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/kegg_seed0_original
Log directory: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/logs/kegg_seed0_original


## Sources

- [DREAMwalk paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC10272215/) and Supplementary Table 4.
- [Author Source Data](https://pmc.ncbi.nlm.nih.gov/articles/instance/10272215/bin/41467_2023_39301_MOESM4_ESM.xlsx), `Figure2d`, rows 2–11.
- [Pinned DREAMwalk](https://github.com/eugenebang/DREAMwalk/tree/2f93723c3fbfa899d0f23c02e3459de8bfac0581)
- [Paper-linked Zenodo release](https://zenodo.org/records/7935342): source and demo archive, without the full processed KEGG benchmark.
- [KEGG API manual](https://www.genome.jp/kegg/rest/keggapi.html)
- [KEGG entry format](https://www.genome.jp/kegg/docs/dbentry.html)
- [KEGG API guidance](https://www.genome.jp/kegg/rest/)

Snapshot provenance is saved with raw responses and the input manifest. Paper-reference extraction is in `references/kegg_fig2d_dreamwalk_provenance.json`.
